## Summary: JavaScript Type Inference Tools

Based on our exploration, here are the available tools and approaches for extracting type information from JavaScript:

### 1. **Existing jy Package Capabilities**
- ✅ **Function name extraction**: `func_name_and_params_pairs()`
- ✅ **Parameter names**: Complete parameter identification
- ✅ **Default values**: Extracts default values from JavaScript
- ✅ **Basic type inference**: Can infer types from default values

### 2. **Type Inference from Default Values**
- ✅ `string = "hello"` → `string`
- ✅ `number = 42` → `number` 
- ✅ `boolean = true` → `boolean`
- ✅ `array = []` → `array`
- ✅ `object = {}` → `object`

### 3. **Type Inference from Parameter Names**
- ✅ Heuristic patterns (e.g., `isActive` → `boolean`, `count` → `number`)
- ✅ Configurable pattern matching
- ✅ Confidence scoring

### 4. **Advanced Features You Can Build**
- ✅ **Enhanced AST analysis** for complex default values
- ✅ **TypeScript interface generation** from JS functions
- ✅ **Confidence scoring** for type predictions
- ✅ **Multiple inference strategies** combined

### 5. **TypeScript Parsing**
- ⚠️ **Partial support**: `jy` has TypeScript parsing (`parse_ts`) but needs setup
- ⚠️ **Dependencies**: Requires `tree-sitter` and `tree-sitter-languages`
- ✅ **Explicit types**: When working, can extract explicit TypeScript type annotations

### 6. **Return Type Inference (Potential)**
- 🔄 **Function body analysis**: Analyze return statements
- 🔄 **Name-based inference**: Function naming patterns
- 🔄 **JSDoc parsing**: Extract `@returns` annotations
- 🔄 **TypeScript annotations**: Parse explicit return types

### 7. **Recommendations**

**For immediate use:**
```python
from jy import func_name_and_params_pairs

# Use enhanced inference system (see definition of JavaScriptTypeInference in notebook below)
type_inferrer = JavaScriptTypeInference()
analysis = type_inferrer.analyze_functions(js_code)
```

**For advanced type extraction:**
1. Use the `JavaScriptTypeInference` class we built
2. Combine default value analysis with name-based heuristics
3. Generate TypeScript interfaces for documentation
4. Add confidence scoring for type predictions

**For TypeScript support:**
```bash
pip install tree-sitter==0.21.3 tree-sitter-languages
```

The `jy` package provides an excellent foundation, and we've shown how to extend it with sophisticated type inference capabilities!

# JavaScript Type Inference and Enhanced Parsing

Let me explore the type information capabilities in JavaScript parsing, building on what we've learned about the `jy` package.

In [ ]:
# Let's create comprehensive JavaScript code with various type scenarios
js_code_with_types = '''
// Function with typed parameters and default values - we can infer types from defaults
function processData(name = "defaultName", age = 25, isActive = true, score = 3.14) {
    return name + " is " + age + " years old";
}

// Arrow function with defaults of different types
const calculateStats = (count = 0, ratio = 0.5, enabled = false, tags = []) => {
    return count * ratio;
};

// Function with mixed parameter types
function configureApp(settings = {}, port = 8080, debug = true, version = "1.0.0") {
    return settings;
}

// More complex defaults
const createUser = (
    profile = { name: "guest", role: "user" },
    permissions = ["read"],
    metadata = null,
    timestamp = Date.now()
) => {
    return profile;
};

// Function with no defaults - harder to infer types
function rawFunction(a, b, c) {
    return a + b + c;
}

// Different ways to declare functions
var oldStyleFunc = function(x = 10, y = "hello") {
    return x + y;
};

let modernFunc = function(flag = true, data = [1, 2, 3]) {
    return flag ? data : [];
};

// Functions assigned to object properties
utils.validator = function(input = "", strict = false) {
    return input.length > 0;
};

obj.nested.helper = function(config = { timeout: 5000, retry: 3 }) {
    return config.timeout;
};
'''

print("JavaScript code with various type scenarios:")
print(js_code_with_types)

JavaScript code with various type scenarios:

// Function with typed parameters and default values - we can infer types from defaults
function processData(name = "defaultName", age = 25, isActive = true, score = 3.14) {
    return name + " is " + age + " years old";
}

// Arrow function with defaults of different types
const calculateStats = (count = 0, ratio = 0.5, enabled = false, tags = []) => {
    return count * ratio;
};

// Function with mixed parameter types
function configureApp(settings = {}, port = 8080, debug = true, version = "1.0.0") {
    return settings;
}

// More complex defaults
const createUser = (
    profile = { name: "guest", role: "user" },
    permissions = ["read"],
    metadata = null,
    timestamp = Date.now()
) => {
    return profile;
};

// Function with no defaults - harder to infer types
function rawFunction(a, b, c) {
    return a + b + c;
}

// Different ways to declare functions
var oldStyleFunc = function(x = 10, y = "hello") {
    return x + y;
};

In [ ]:
# Extract function names and parameters using jy
from jy import func_name_and_params_pairs

print("=== Basic Function Extraction (Current jy capability) ===")
for func_name, params in func_name_and_params_pairs(js_code_with_types):
    print(f"\nFunction: {func_name}")
    for param in params:
        print(f"  - {param}")

=== Basic Function Extraction (Current jy capability) ===

Function: processData
  - {'name': 'name', 'default': 'defaultName'}
  - {'name': 'age', 'default': 25}
  - {'name': 'isActive', 'default': True}
  - {'name': 'score', 'default': 3.14}

Function: calculateStats
  - {'name': 'count', 'default': 0}
  - {'name': 'ratio', 'default': 0.5}
  - {'name': 'enabled', 'default': False}
  - {'name': 'tags', 'default': None}

Function: configureApp
  - {'name': 'settings', 'default': None}
  - {'name': 'port', 'default': 8080}
  - {'name': 'debug', 'default': True}
  - {'name': 'version', 'default': '1.0.0'}

Function: createUser
  - {'name': 'profile', 'default': None}
  - {'name': 'permissions', 'default': None}
  - {'name': 'metadata', 'default': None}
  - {'name': 'timestamp', 'default': None}

Function: rawFunction
  - {'name': 'a'}
  - {'name': 'b'}
  - {'name': 'c'}

Function: oldStyleFunc
  - {'name': 'x', 'default': 10}
  - {'name': 'y', 'default': 'hello'}

Function: modernFunc
  

In [ ]:
# Now let's create a type inference system that builds on jy
def infer_type_from_default(default_value):
    """Infer JavaScript type from default value"""
    if default_value is None:
        return "any"  # Could be null, undefined, or not parseable
    elif isinstance(default_value, bool):
        return "boolean"
    elif isinstance(default_value, int):
        return "number"  # JavaScript doesn't distinguish int from float
    elif isinstance(default_value, float):
        return "number"
    elif isinstance(default_value, str):
        return "string"
    elif isinstance(default_value, list):
        return "array"
    elif isinstance(default_value, dict):
        return "object"
    else:
        return "unknown"

def infer_type_from_name(param_name):
    """Infer type from parameter name patterns (heuristic approach)"""
    name_lower = param_name.lower()
    
    # Common naming patterns
    if any(word in name_lower for word in ['is', 'has', 'can', 'should', 'enabled', 'active', 'flag', 'debug']):
        return "boolean"
    elif any(word in name_lower for word in ['count', 'size', 'length', 'age', 'port', 'timeout', 'retry']):
        return "number"
    elif any(word in name_lower for word in ['name', 'title', 'message', 'text', 'input', 'output', 'version']):
        return "string"
    elif any(word in name_lower for word in ['list', 'items', 'tags', 'permissions', 'data']):
        return "array"
    elif any(word in name_lower for word in ['config', 'settings', 'options', 'profile', 'metadata']):
        return "object"
    else:
        return "any"

def enhanced_func_parsing(js_code):
    """Enhanced function parsing with type inference"""
    functions = []
    
    for func_name, params in func_name_and_params_pairs(js_code):
        enhanced_params = []
        
        for param in params:
            enhanced_param = param.copy()
            
            # Try to infer type from default value first
            if 'default' in param:
                enhanced_param['type'] = infer_type_from_default(param['default'])
                enhanced_param['type_source'] = 'default_value'
            else:
                # Fall back to name-based inference
                enhanced_param['type'] = infer_type_from_name(param['name'])
                enhanced_param['type_source'] = 'name_heuristic'
            
            enhanced_params.append(enhanced_param)
        
        functions.append({
            'name': func_name,
            'parameters': enhanced_params
        })
    
    return functions

print("=== Enhanced Function Parsing with Type Inference ===")
enhanced_functions = enhanced_func_parsing(js_code_with_types)

for func in enhanced_functions:
    print(f"\nFunction: {func['name']}")
    for param in func['parameters']:
        type_info = f"({param['type']} - {param['type_source']})"
        if 'default' in param:
            print(f"  - {param['name']}: {type_info} = {param['default']}")
        else:
            print(f"  - {param['name']}: {type_info}")

=== Enhanced Function Parsing with Type Inference ===

Function: processData
  - name: (string - default_value) = defaultName
  - age: (number - default_value) = 25
  - isActive: (boolean - default_value) = True
  - score: (number - default_value) = 3.14

Function: calculateStats
  - count: (number - default_value) = 0
  - ratio: (number - default_value) = 0.5
  - enabled: (boolean - default_value) = False
  - tags: (any - default_value) = None

Function: configureApp
  - settings: (any - default_value) = None
  - port: (number - default_value) = 8080
  - debug: (boolean - default_value) = True
  - version: (string - default_value) = 1.0.0

Function: createUser
  - profile: (any - default_value) = None
  - permissions: (any - default_value) = None
  - metadata: (any - default_value) = None
  - timestamp: (any - default_value) = None

Function: rawFunction
  - a: (any - name_heuristic)
  - b: (any - name_heuristic)
  - c: (any - name_heuristic)

Function: oldStyleFunc
  - x: (number - d

In [ ]:
# Let's look at TypeScript parsing for explicit type information
typescript_code = '''
interface UserProfile {
    name: string;
    age: number;
    isActive: boolean;
    tags: string[];
    metadata?: object;
}

function processUser(profile: UserProfile, options: {strict: boolean, timeout: number}): string {
    return profile.name;
}

const calculateScore = (
    baseScore: number, 
    multiplier: number = 1.0, 
    bonus?: number
): number => {
    return baseScore * multiplier + (bonus || 0);
};

// Mixed TypeScript and JavaScript
function mixedFunction(
    id: string,
    count = 10,  // Type inferred from default
    enabled: boolean = true,
    data?: any[]
): Promise<string> {
    return Promise.resolve(id);
}
'''

print("TypeScript code with explicit types:")
print(typescript_code)

TypeScript code with explicit types:

interface UserProfile {
    name: string;
    age: number;
    isActive: boolean;
    tags: string[];
    metadata?: object;
}

function processUser(profile: UserProfile, options: {strict: boolean, timeout: number}): string {
    return profile.name;
}

const calculateScore = (
    baseScore: number, 
    multiplier: number = 1.0, 
    bonus?: number
): number => {
    return baseScore * multiplier + (bonus || 0);
};

// Mixed TypeScript and JavaScript
function mixedFunction(
    id: string,
    count = 10,  // Type inferred from default
    enabled: boolean = true,
    data?: any[]
): Promise<string> {
    return Promise.resolve(id);
}



In [ ]:
# Let's try the TypeScript parsing functions from jy
try:
    from jy import parse_ts
    print("=== TypeScript Parsing Available ===")
    
    # This requires tree-sitter which may not be installed
    ts_items = list(parse_ts(typescript_code))
    for name, info in ts_items:
        print(f"Name: {name}")
        print(f"Info: {info}")
        print()
        
except ImportError as e:
    print(f"TypeScript parsing not available: {e}")
    print("You would need to install: pip install tree-sitter tree-sitter-languages")
except Exception as e:
    print(f"Error in TypeScript parsing: {e}")
    print("TypeScript parsing might require additional setup")

=== TypeScript Parsing Available ===
Error in TypeScript parsing: __init__() takes exactly 1 argument (2 given)
TypeScript parsing might require additional setup


In [ ]:
# Let's create a more advanced type inference system using the AST from esprima
from jy.js_parse import parse_js
import esprima

def extract_complex_defaults(js_code):
    """Extract more detailed information about default values from the AST"""
    ast = parse_js(js_code)
    
    def analyze_literal(node):
        """Analyze a literal node to get its type and value"""
        if hasattr(node, 'type'):
            if node.type == 'Literal':
                value = node.value
                if isinstance(value, bool):
                    return {'type': 'boolean', 'value': value}
                elif isinstance(value, (int, float)):
                    return {'type': 'number', 'value': value}
                elif isinstance(value, str):
                    return {'type': 'string', 'value': value}
                elif value is None:
                    return {'type': 'null', 'value': None}
            elif node.type == 'ArrayExpression':
                return {'type': 'array', 'value': 'array'}
            elif node.type == 'ObjectExpression':
                return {'type': 'object', 'value': 'object'}
            elif node.type == 'CallExpression':
                # Handle things like Date.now()
                if (hasattr(node, 'callee') and 
                    hasattr(node.callee, 'type') and 
                    node.callee.type == 'MemberExpression'):
                    return {'type': 'function_call', 'value': 'computed'}
        return {'type': 'unknown', 'value': 'unknown'}
    
    def extract_function_info(node):
        """Extract function information with enhanced default analysis"""
        if node.type in ['FunctionDeclaration', 'FunctionExpression', 'ArrowFunctionExpression']:
            func_name = None
            if hasattr(node, 'id') and node.id:
                func_name = node.id.name
                
            params = []
            if hasattr(node, 'params'):
                for param in node.params:
                    param_info = {'name': None, 'type': 'any', 'optional': False}
                    
                    if param.type == 'Identifier':
                        param_info['name'] = param.name
                    elif param.type == 'AssignmentPattern':
                        # Parameter with default value
                        param_info['name'] = param.left.name
                        param_info['optional'] = True
                        default_analysis = analyze_literal(param.right)
                        param_info['type'] = default_analysis['type']
                        param_info['default'] = default_analysis['value']
                    
                    params.append(param_info)
            
            return {'name': func_name, 'params': params, 'node_type': node.type}
        return None
    
    def traverse_ast(node, functions):
        """Traverse the AST to find function declarations"""
        if hasattr(node, 'type'):
            func_info = extract_function_info(node)
            if func_info and func_info['name']:
                functions.append(func_info)
        
        # Traverse child nodes
        if hasattr(node, 'body'):
            if isinstance(node.body, list):
                for child in node.body:
                    traverse_ast(child, functions)
            else:
                traverse_ast(node.body, functions)
        
        # Handle other node properties that might contain child nodes
        for attr_name in ['expression', 'left', 'right', 'init', 'declarations']:
            if hasattr(node, attr_name):
                attr_value = getattr(node, attr_name)
                if attr_value:
                    if isinstance(attr_value, list):
                        for item in attr_value:
                            traverse_ast(item, functions)
                    else:
                        traverse_ast(attr_value, functions)
    
    functions = []
    traverse_ast(ast, functions)
    return functions

# Test our enhanced analysis
print("=== Enhanced AST-based Type Analysis ===")
enhanced_analysis = extract_complex_defaults(js_code_with_types)

for func_info in enhanced_analysis:
    print(f"\nFunction: {func_info['name']} ({func_info['node_type']})")
    for param in func_info['params']:
        if 'default' in param:
            print(f"  - {param['name']}: {param['type']} = {param['default']}")
        else:
            print(f"  - {param['name']}: {param['type']}")

=== Enhanced AST-based Type Analysis ===

Function: processData (FunctionDeclaration)
  - name: string = defaultName
  - age: number = 25
  - isActive: boolean = True
  - score: number = 3.14

Function: configureApp (FunctionDeclaration)
  - settings: object = object
  - port: number = 8080
  - debug: boolean = True
  - version: string = 1.0.0

Function: rawFunction (FunctionDeclaration)
  - a: any
  - b: any
  - c: any


In [ ]:
# Let's create a complete type inference system that combines multiple approaches
class JavaScriptTypeInference:
    """
    A comprehensive JavaScript type inference system that combines:
    1. Default value analysis
    2. Parameter name heuristics  
    3. AST analysis
    4. Pattern recognition
    """
    
    def __init__(self):
        self.type_patterns = {
            'boolean': ['is', 'has', 'can', 'should', 'enabled', 'active', 'flag', 'debug', 'strict'],
            'number': ['count', 'size', 'length', 'age', 'port', 'timeout', 'retry', 'index', 'id', 'score'],
            'string': ['name', 'title', 'message', 'text', 'input', 'output', 'version', 'url', 'path'],
            'array': ['list', 'items', 'tags', 'permissions', 'data', 'values', 'results'],
            'object': ['config', 'settings', 'options', 'profile', 'metadata', 'params']
        }
    
    def infer_from_default(self, default_value):
        """Infer type from default value"""
        if default_value is None:
            return "any"
        elif isinstance(default_value, bool):
            return "boolean"
        elif isinstance(default_value, (int, float)):
            return "number"
        elif isinstance(default_value, str):
            return "string"
        elif isinstance(default_value, list):
            return "array"
        elif isinstance(default_value, dict):
            return "object"
        else:
            return "unknown"
    
    def infer_from_name(self, param_name):
        """Infer type from parameter name using patterns"""
        name_lower = param_name.lower()
        
        for type_name, patterns in self.type_patterns.items():
            if any(pattern in name_lower for pattern in patterns):
                return type_name
        
        return "any"
    
    def analyze_functions(self, js_code):
        """Complete function analysis with type inference"""
        from jy import func_name_and_params_pairs
        
        functions = []
        
        for func_name, params in func_name_and_params_pairs(js_code):
            enhanced_function = {
                'name': func_name,
                'parameters': [],
                'metadata': {
                    'total_params': len(params),
                    'optional_params': sum(1 for p in params if 'default' in p),
                    'required_params': sum(1 for p in params if 'default' not in p)
                }
            }
            
            for param in params:
                enhanced_param = {
                    'name': param['name'],
                    'required': 'default' not in param,
                    'type_inference': {}
                }
                
                # Add default value if present
                if 'default' in param:
                    enhanced_param['default'] = param['default']
                
                # Multiple type inference approaches
                if 'default' in param:
                    default_type = self.infer_from_default(param['default'])
                    enhanced_param['type_inference']['from_default'] = default_type
                    enhanced_param['type'] = default_type  # Primary type
                else:
                    enhanced_param['type'] = "any"
                
                name_type = self.infer_from_name(param['name'])
                enhanced_param['type_inference']['from_name'] = name_type
                
                # If no default, use name-based inference as primary
                if enhanced_param['type'] == "any" and name_type != "any":
                    enhanced_param['type'] = name_type
                
                # Confidence scoring
                confidence = 0.9 if 'default' in param else 0.3
                if enhanced_param['type_inference']['from_name'] == enhanced_param['type']:
                    confidence += 0.2
                enhanced_param['type_confidence'] = min(confidence, 1.0)
                
                enhanced_function['parameters'].append(enhanced_param)
            
            functions.append(enhanced_function)
        
        return functions
    
    def generate_typescript_interface(self, function_analysis):
        """Generate TypeScript interface definitions from analysis"""
        interfaces = []
        
        for func in function_analysis:
            interface_name = f"{func['name'].replace('.', '_').title()}Params"
            
            interface_def = f"interface {interface_name} {{\n"
            for param in func['parameters']:
                optional_marker = "?" if not param['required'] else ""
                interface_def += f"  {param['name']}{optional_marker}: {param['type']};\n"
            interface_def += "}"
            
            interfaces.append({
                'function': func['name'],
                'interface_name': interface_name,
                'definition': interface_def
            })
        
        return interfaces

# Test our comprehensive system
type_inferrer = JavaScriptTypeInference()
comprehensive_analysis = type_inferrer.analyze_functions(js_code_with_types)

print("=== Comprehensive Type Analysis ===")
for func in comprehensive_analysis:
    print(f"\nFunction: {func['name']}")
    print(f"Metadata: {func['metadata']}")
    for param in func['parameters']:
        req_str = "required" if param['required'] else "optional"
        conf_str = f"(confidence: {param['type_confidence']:.1f})"
        if 'default' in param:
            print(f"  - {param['name']}: {param['type']} = {param['default']} [{req_str}] {conf_str}")
        else:
            print(f"  - {param['name']}: {param['type']} [{req_str}] {conf_str}")
        print(f"    Type inference: {param['type_inference']}")

=== Comprehensive Type Analysis ===

Function: processData
Metadata: {'total_params': 4, 'optional_params': 4, 'required_params': 0}
  - name: string = defaultName [optional] (confidence: 1.0)
    Type inference: {'from_default': 'string', 'from_name': 'string'}
  - age: number = 25 [optional] (confidence: 1.0)
    Type inference: {'from_default': 'number', 'from_name': 'number'}
  - isActive: boolean = True [optional] (confidence: 1.0)
    Type inference: {'from_default': 'boolean', 'from_name': 'boolean'}
  - score: number = 3.14 [optional] (confidence: 1.0)
    Type inference: {'from_default': 'number', 'from_name': 'number'}

Function: calculateStats
Metadata: {'total_params': 4, 'optional_params': 4, 'required_params': 0}
  - count: number = 0 [optional] (confidence: 1.0)
    Type inference: {'from_default': 'number', 'from_name': 'number'}
  - ratio: number = 0.5 [optional] (confidence: 0.9)
    Type inference: {'from_default': 'number', 'from_name': 'any'}
  - enabled: boolean =

In [ ]:
# Generate TypeScript interfaces from our analysis
print("\n=== Generated TypeScript Interfaces ===")
typescript_interfaces = type_inferrer.generate_typescript_interface(comprehensive_analysis)

for interface in typescript_interfaces[:3]:  # Show first 3 to keep output manageable
    print(f"\nFor function: {interface['function']}")
    print(interface['definition'])

# Show return type inference potential
print("\n=== Return Type Inference Ideas ===")
print("""
For return type inference, we could analyze:

1. Return statements in function bodies:
   - return "string" → string
   - return 42 → number  
   - return true → boolean
   - return [] → array
   - return {} → object

2. Function name patterns:
   - Functions starting with 'is', 'has', 'can' → boolean
   - Functions with 'count', 'calculate' → number
   - Functions with 'get', 'find' → depends on context
   - Functions with 'create', 'build' → object

3. JSDoc comments:
   - @returns {string} → string
   - @returns {Promise<User>} → Promise<User>

4. TypeScript annotations:
   - function foo(): string → string
   - async function bar(): Promise<number> → Promise<number>
""")


=== Generated TypeScript Interfaces ===

For function: processData
interface ProcessdataParams {
  name?: string;
  age?: number;
  isActive?: boolean;
  score?: number;
}

For function: calculateStats
interface CalculatestatsParams {
  count?: number;
  ratio?: number;
  enabled?: boolean;
  tags?: array;
}

For function: configureApp
interface ConfigureappParams {
  settings?: object;
  port?: number;
  debug?: boolean;
  version?: string;
}

=== Return Type Inference Ideas ===

For return type inference, we could analyze:

1. Return statements in function bodies:
   - return "string" → string
   - return 42 → number  
   - return true → boolean
   - return [] → array
   - return {} → object

2. Function name patterns:
   - Functions starting with 'is', 'has', 'can' → boolean
   - Functions with 'count', 'calculate' → number
   - Functions with 'get', 'find' → depends on context
   - Functions with 'create', 'build' → object

3. JSDoc comments:
   - @returns {string} → string
   - @

In [ ]:
import rh